# UAV Disaster Surveillance — Genetic Algorithm Strategy Engine
**Artificial Intelligence · BSCS 5 · Assignment 2**

This notebook is an interactive companion to the modular `uav_ga/` source code. It plans fleet deployment/waypoints, speed and sensor power; plots optimal routes and target coverage; and compares genetic search with a matched-budget random-search baseline.

**To run locally:** open this notebook from the extracted project folder. **To run on Google Colab:** first upload the complete project to GitHub, set `GITHUB_REPO_URL` in the next cell to your repository's HTTPS clone URL and run all cells.


In [ ]:
# SETUP — installs required packages and loads source from local folder or GitHub.
import subprocess, sys, os
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24,<3', 'matplotlib>=3.7,<4'])
GITHUB_REPO_URL = ''  # Colab example: 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'
project = Path.cwd()
if not (project / 'uav_ga').is_dir():
    if not GITHUB_REPO_URL:
        raise RuntimeError('Not inside the project folder. On Google Colab, enter your GITHUB_REPO_URL above and rerun this cell.')
    clone_dir = Path('/content/uav_ga_assignment')
    if not (clone_dir / 'uav_ga').is_dir():
        subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO_URL, str(clone_dir)])
    project = clone_dir
os.chdir(project)
sys.path.insert(0, str(project))
print('Project loaded:', project)


## 1. Configure a reproducible hazard-grid mission
Coordinates are `(x,y)` and the default grid has 50 × 50 cells. The model uses 3 UAVs, priority/disaster regions, target cells, weather/hazard pockets and no-fly obstacles.

In [ ]:
from uav_ga.config import MissionConfig
from uav_ga.environment import HazardGrid
from uav_ga.ga import GeneticAlgorithm
from uav_ga.baseline import random_search
from uav_ga.visualization import render_simulation
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

config = MissionConfig.from_json('config/mission.json')
# For a shorter demonstration, uncomment the next line:
# config = config.with_changes(population_size=12, generations=5, stagnation_patience=5)
world = HazardGrid.generate(config)
print(f'Grid: {config.grid_size}x{config.grid_size}; UAVs: {config.drones}; targets: {len(world.targets)}')


## 2. Run genetic optimization and fair-budget baseline
Each chromosome stores aligned waypoint coordinates, speed adjustment, and sensor power segments. Tournament selection + elitism, geometric/tail crossover, spatial mutation, feasibility repair and penalties optimize the mission.

In [ ]:
ga = GeneticAlgorithm(config, world).run()
baseline = random_search(config, world, ga.evaluations)
print('GA fitness:', round(ga.fitness.score, 6))
print('Random-search fitness:', round(baseline.best_score, 6))
print('Difference (GA - random):', round(ga.fitness.score - baseline.best_score, 6))
print('Equal fitness evaluations:', ga.evaluations)
print('Coverage:', round(ga.fitness.coverage, 4), 'Target detection:', round(ga.fitness.detection, 4))
print('Energy normalized:', round(ga.fitness.energy, 4), 'Max time normalized:', round(ga.fitness.mission_time, 4))
print('Hazard exposure:', round(ga.fitness.risk, 4), 'Feasible:', ga.fitness.feasible)


## 3. Visualize fitness, path routes, weighted coverage, evolution
All figures are saved into `outputs/colab_run/` so they can be reused in presentations or the assignment submission.

In [ ]:
out_dir = Path('outputs/colab_run')
render_simulation(config, world, ga, out_dir)
for filename in ['fitness_convergence.png', 'uav_paths.png', 'coverage_heatmap.png', 'evolution_over_generations.png']:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.imshow(mpimg.imread(out_dir / filename))
    ax.axis('off')
    ax.set_title(filename.replace('_', ' ').replace('.png', '').title())
    plt.show()


## 4. Hyperparameter sensitivity (optional)
Sweep population size, mutation rate, and tournament selection pressure. The recorded run counts/fitness may differ from the example report because this is a fresh execution. The `quick=True` setting uses 12 generations for fast demonstration.

In [ ]:
from uav_ga.experiments import sensitivity_analysis
# Set to True to perform nine fresh sensitivity experiments.
RUN_SENSITIVITY = False
if RUN_SENSITIVITY:
    rows = sensitivity_analysis(config, 'outputs/colab_sensitivity', quick=True)
    for r in rows:
        print(r['parameter'], r['value'], r['best_fitness'], r['evaluations'])
    display(__import__('IPython').display.Image(filename='outputs/colab_sensitivity/sensitivity.png'))
else:
    print('Sensitivity experiment skipped; set RUN_SENSITIVITY=True to run it.')


**Academic note:** This is a synthetic disaster map, not a live UAV controller. Metrics are deterministic for fixed package versions/seed, and results from one random seed do not prove statistical superiority. See `REPORT.docx` for the complete methodology and limitations.